# TinyTrees / TreeMatch — a FiftyOne demo

This notebook builds a FiftyOne dataset out of the **TinyTrees** benchmark
(dgominski/tinytrees) and the pretrained **TreeMatch** models
(dgominski/treematch), so you can visually explore:

- per-tree **point annotations** across three regions/sensors (Rwanda /
  PlanetScope, China / Gaofen-2, France / SPOT-6)
- **noisy weak supervision** vs. **strong (expert) supervision** side by side
- pretrained TreeMatch **density-map predictions** overlaid on ground truth
- **count error** per tile, sortable/filterable in the App
- a **cross-sensor embedding view** to see domain shift between regions
- tiles plotted on the **Map panel** (data is georeferenced)

### Assumptions baked into this notebook (please verify before trusting numbers)

1. You're running this **inside a dedicated virtual environment** with
   FiftyOne **1.17** installed in it, and the kernel registered for Jupyter
   (see Section 0 for a copy-pasteable setup). The notebook uses whatever
   directory you launch Jupyter from as its workspace root — no hardcoded
   path required.
2. The `treematch` GitHub repo is cloned locally so we can reuse the
   **official dataset loader classes** (`data.ps.PlanetScopeStrong`, etc.)
   rather than re-deriving the GeoTIFF→tensor and point-extraction logic
   ourselves. This matters because the loaders already handle band
   normalization and the exact pixel grid the points are annotated on —
   re-implementing that from the raw GeoTIFF + GeoPackage is more error-prone.
3. TinyTrees is **~21 GB** as a single `.zip` on Hugging Face. There's no way
   to avoid downloading that whole archive at least once (HF hosts it as one
   blob), but we avoid unpacking all of it to disk — we open the zip and
   selectively extract only a sampled subset of tiles.
4. **License is CC BY-NC 4.0** — research/education only, not commercial use.
5. Wherever the exact loader-class API (return types, attribute names for
   file paths, etc.) matters and I can't verify it without running the repo
   locally, I've flagged it with `# TODO: verify against repo source` so you
   know exactly where to look if something doesn't line up.

### Recommended path through this notebook

Run **Section 1 (synthetic sanity check)** first — it builds a tiny fake
dataset with the same schema (keypoints, heatmaps, geolocation, scalar
fields) purely in-memory, so you can confirm your FiftyOne install/App is
wired up correctly *before* waiting on a 21 GB download. Then move on to the
real pipeline in Sections 2+.


## 0. Environment setup (run once, in a terminal — not in the notebook)

This section intentionally installs only what's needed to *get the notebook
running* — fiftyone, Jupyter, and the huggingface_hub client for the
download step. **Everything TreeMatch-specific (torch, albumentations,
segmentation-models-pytorch, rasterio, geopandas, etc.) gets installed in
Section 2, straight from the repo's own `requirements.txt`**, right after
cloning it — that way we're always matching the exact versions the repo was
built against, instead of a hand-maintained duplicate list drifting out of
sync.

```bash
# Pick any workspace directory you like — everything this notebook creates
# (cloned repo, downloaded data, generated thumbnails) lands under here.
mkdir -p ~/fiftyone-tinytrees-demo
cd ~/fiftyone-tinytrees-demo

python3 -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate

pip install "fiftyone==1.17.*"
pip install huggingface_hub ipykernel

python -m ipykernel install --user \
    --name fiftyone-tinytrees-demo \
    --display-name "FiftyOne TinyTrees Demo"
```

Then launch Jupyter **from this same directory** (`jupyter lab` or
`jupyter notebook`, or open this file from here in VS Code/Cursor) — the
config cell below uses your current working directory as the workspace
root, so starting Jupyter from elsewhere would put everything in the wrong
place.

**Verify what's actually installed before going further** — run this in a
terminal with the venv active:

```bash
python -c "import fiftyone as fo; print(fo.__version__)"
```

If that doesn't print `1.17.x`, something else in this venv (an existing
install, a conflicting requirement pulled in later, etc.) is overriding the
pin above — resolve that before running Section 2 below, since the rest of
the notebook assumes whatever version prints here is the one actually
running.

In Jupyter (or VS Code / Cursor's notebook UI), select the
**"FiftyOne TinyTrees Demo"** kernel before running the cells below.

> Apple Silicon note: `torch` isn't installed until Section 2 (via the
> repo's own `requirements.txt`), and device detection (`mps`/`cuda`/`cpu`)
> happens right after that install, not in the early config cell — so the
> notebook is safe to run top-to-bottom in order. If you'd rather have torch
> available immediately for some other reason, `pip install torch
> torchvision` now is harmless; Section 2's install will just see it already
> satisfied.


In [ ]:
import os
import sys
import json
import random
import zipfile
import subprocess
from pathlib import Path

import numpy as np

import fiftyone as fo
import fiftyone.brain as fob
from fiftyone import ViewField as F

print("FiftyOne version:", fo.__version__)
if not fo.__version__.startswith("1.17"):
    print(f"  [heads up] expected 1.17.x, got {fo.__version__} — App/panel behavior "
          "referenced in this notebook was checked against 1.17 and may differ.")

# ------------------------------------------------------------------
# Paths — defaults to wherever you launched Jupyter from (see Section 0).
# Override ROOT explicitly if you'd rather point this at a different
# directory than your current working one.
# ------------------------------------------------------------------
ROOT = Path.cwd()
REPO_DIR = ROOT / "treematch"                 # git clone of dgominski/treematch
DATA_DIR = ROOT / "data" / "tinytrees"        # extracted TinyTrees subset
MEDIA_DIR = ROOT / "media" / "tinytrees"      # generated RGB thumbnails + heatmap PNGs
CACHE_DIR = ROOT / ".hf_cache"                # huggingface_hub download cache

for d in (DATA_DIR, MEDIA_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "tinytrees-demo"

# ------------------------------------------------------------------
# Subsample sizes — keep small for a snappy demo; bump up later
# ------------------------------------------------------------------
REGIONS = {
    # key -> (hf/zip prefix, sensor tag, TreeMatch hub subfolder)
    "ps":   {"name": "rwanda",  "sensor": "PlanetScope", "hub_subfolder": "ps"},
    "gf":   {"name": "china",   "sensor": "Gaofen-2",    "hub_subfolder": "gf"},
    "spot": {"name": "france",  "sensor": "SPOT-6",      "hub_subfolder": "spot"},
}
SPLITS = ["train_strong", "train_weak", "test"]
N_TILES_PER_SPLIT = 15   # per region, per split — tune this

# The loader classes (PlanetScopeStrong/GaofenStrong/SPOTStrong) always crop
# each tile to a fixed IMSIZE x IMSIZE window (RandomCrop for train splits,
# CenterCrop for test) — passing imsize=None breaks their internal
# albumentations pipeline. Must be divisible by 32 (ResNet-50 U-Net has 5
# downsampling stages); 64/128/256/512 are all safe. Bigger = more context
# per tile and a nicer-looking demo image, at the cost of a smaller
# effective random-crop offset range for train splits.
IMSIZE = 256

RANDOM_SEED = 51
random.seed(RANDOM_SEED)

# Section 3 checks the local DATA_DIR before touching the network/zip at
# all — set this to True to force re-fetching a fresh random subset even
# if tiles already appear to be staged locally.
FORCE_REFETCH = False


## 1. Sanity check with synthetic data (run this first, no downloads needed)

This builds a throwaway dataset with the exact schema we'll use for the real
data — `Keypoints` for ground-truth tree points, `Heatmap` for predicted tree
density, scalar count/error fields, region/sensor/split tags, and a
`GeoLocation` field for the Map panel. If this section looks right in the
App, the rest of the notebook is just "same schema, real data."


In [ ]:
def make_synthetic_sample(region_key, split, idx, size=256):
    """Fabricate one fake tile with random points + a random density heatmap."""
    region = REGIONS[region_key]

    # fake RGB thumbnail
    img = (np.random.rand(size, size, 3) * 255).astype(np.uint8)
    from PIL import Image
    img_path = MEDIA_DIR / f"synthetic_{region_key}_{split}_{idx}.png"
    Image.fromarray(img).save(img_path)

    # fake ground-truth tree points (normalized [0,1] coords, FiftyOne convention)
    n_trees = random.randint(20, 200)
    xs = np.random.rand(n_trees)
    ys = np.random.rand(n_trees)
    points = list(zip(xs.tolist(), ys.tolist()))

    # fake predicted density map + fake predicted count
    density = np.random.rand(size, size).astype(np.float32)
    density_png_path = MEDIA_DIR / f"synthetic_{region_key}_{split}_{idx}_density.png"
    norm = (density - density.min()) / (density.max() - density.min() + 1e-8)
    Image.fromarray((norm * 255).astype(np.uint8)).save(density_png_path)
    count_pred = float(density.sum() / density.size * n_trees * random.uniform(0.7, 1.3))

    sample = fo.Sample(filepath=str(img_path))
    sample["ground_truth_points"] = fo.Keypoints(
        keypoints=[fo.Keypoint(label="tree", points=points)]
    )
    sample["density_pred"] = fo.Heatmap(map_path=str(density_png_path))
    sample["region"] = region["name"]
    sample["sensor"] = region["sensor"]
    sample["split"] = split
    sample["count_gt"] = n_trees
    sample["count_pred"] = count_pred
    sample["count_error"] = count_pred - n_trees
    sample["count_abs_error"] = abs(count_pred - n_trees)
    # fake but plausible lon/lat per region, jittered
    base_lonlat = {"rwanda": (29.9, -1.9), "china": (113.3, 23.1), "france": (2.3, 46.6)}
    lon, lat = base_lonlat[region["name"]]
    sample["location"] = fo.GeoLocation(
        point=[lon + random.uniform(-0.5, 0.5), lat + random.uniform(-0.5, 0.5)]
    )
    sample.tags = [region["name"], split]
    return sample


if fo.dataset_exists("tinytrees-synthetic-sanity-check"):
    fo.delete_dataset("tinytrees-synthetic-sanity-check")
sanity_ds = fo.Dataset("tinytrees-synthetic-sanity-check")

samples = []
for region_key in REGIONS:
    for split in SPLITS:
        for i in range(5):
            samples.append(make_synthetic_sample(region_key, split, i))
sanity_ds.add_samples(samples)

print(sanity_ds)


In [ ]:
# Launch the App and eyeball it: keypoints should render as dots on the image,
# density_pred should render as a heatmap overlay, and region/split/count_*
# fields should show up in the sidebar for filtering.
session = fo.launch_app(sanity_ds)


If that looks right — dots for ground-truth trees, a heatmap you can toggle,
scalar fields you can filter/sort on in the sidebar — the schema is solid.
Close/ignore this synthetic dataset and move on to the real pipeline below.


## 2. Clone the TreeMatch repo (model code + official dataset loaders)

We need this for:
- `hub_model.UNetR50.from_pretrained(...)` — the pretrained TreeMatch checkpoints
- `data.ps.PlanetScopeStrong`/`PlanetScopeWeak` (and the `Gaofen*`/`SPOT*`
  equivalents) — the official loaders, so tile→tensor→point-mask logic
  matches exactly what the model was trained/evaluated on. Note the split:
  `*Strong` only accepts `split in {"train_strong", "test"}` (it asserts
  this itself); the sibling `*Weak` class handles `train_weak` instead —
  they're not one class with a flexible `split` argument.

This clones the repo, then installs **the repo's own `requirements.txt`**
(rather than the hand-picked list from Section 0) so we get the exact
versions it was built against — `albumentations`, `timm`,
`segmentation-models-pytorch==0.5.0`, `ultralytics`, `hydra-core`/`omegaconf`,
`wandb`, `tensorboardX`, `seaborn`, `pyogrio`, and pinned `torch`/
`huggingface_hub`/`safetensors` versions. Only *then* do we try importing
from it — installing and importing in separate cells avoids the case where
pip installs something the current kernel process doesn't pick up until
you'd otherwise need a restart.

> Version-pin note: `requirements.txt` pins several packages to specific
> 2023/2024-era versions (`torch==2.1.2`, `torchvision==0.16.2`,
> `rasterio==1.3.8`, `pyogrio==0.12.1`). Depending on your Python version,
> PyPI may not have wheels for those exact versions anymore — and an exact
> `==` pin won't accept an already-installed newer version as satisfying it
> either, so pip fails outright rather than using what you have. The install
> cell below writes a sanitized copy of `requirements.txt` with just those
> version pins stripped (everything else stays pinned exactly as the repo
> specifies) and installs from that instead.
>
> `rasterio`/`pyogrio` are a second, unrelated failure mode on top of that:
> they wrap the GDAL C library, so if pip can't find a prebuilt wheel it
> falls back to compiling from source — which needs GDAL's headers already
> installed on your machine, not just a newer Python package version. The
> install cell checks for `gdal-config` first and tells you to
> `brew install gdal` if it's missing, since unpinning the Python package
> alone won't fix a missing system library.
>
> If some *other* pinned package in this list hits either failure mode, the
> fix is the same each time — add it to the `re.match(...)` pattern in that
> cell.


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/dgominski/treematch.git", str(REPO_DIR)],
        check=True,
    )
else:
    print(f"{REPO_DIR} already exists, skipping clone")

sys.path.insert(0, str(REPO_DIR))


In [ ]:
# Install the repo's OWN pinned dependencies rather than hand-maintaining a
# second copy of this list — data/ps.py, hub_model.py etc. import things like
# albumentations, timm, ultralytics, hydra-core/omegaconf, wandb, tensorboardX,
# seaborn, pyogrio, and a specific segmentation-models-pytorch/huggingface_hub/
# safetensors version, none of which we installed in Section 0.
#
# The repo pins torch==2.1.2 / torchvision==0.16.2 (late-2023 versions). If
# your venv's Python is newer than what those had wheels for, pip will
# refuse the exact pin outright — and critically, it does the same even if
# you already have a newer, perfectly good torch installed, because == means
# == regardless of what's already satisfied. So: strip the version pin on
# just those two lines and let pip pick whatever's compatible with your
# Python/platform; leave every other pin untouched.
import re

req_file = REPO_DIR / "requirements.txt"
sanitized_req_file = ROOT / "requirements_patched.txt"

original_lines = req_file.read_text().splitlines()
patched_lines = []
for line in original_lines:
    m = re.match(r"^(torch|torchvision|rasterio|pyogrio)==", line.strip())
    if m:
        print(f"  [patched] {line.strip()!r} -> {m.group(1)!r} (unpinned)")
        patched_lines.append(m.group(1))
    else:
        patched_lines.append(line)
sanitized_req_file.write_text("\n".join(patched_lines) + "\n")

# rasterio and pyogrio wrap GDAL — if pip can't find a prebuilt wheel for
# your Python/platform it falls back to compiling from source, which needs
# the GDAL system library (headers + `gdal-config`) already installed.
# Unpinning the Python package version (above) doesn't help with that half
# of the problem. Check now rather than after another failed build:
gdal_check = subprocess.run(["gdal-config", "--version"], capture_output=True, text=True)
if gdal_check.returncode != 0:
    print(
        "[heads up] `gdal-config` not found — if the install below fails on "
        "rasterio/pyogrio specifically, install GDAL via Homebrew first:\n"
        "    brew install gdal\n"
        "then re-run this cell."
    )
else:
    print(f"Found GDAL: {gdal_check.stdout.strip()}")

print(f"Installing from {sanitized_req_file} ...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(sanitized_req_file)],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError(
        "pip install failed — see stderr above. If it's another exact-pinned "
        "package with no wheel for your Python/platform (same failure mode as "
        "torch above), add it to the `re.match(...)` pattern in this cell and "
        "re-run — the fix is the same each time: unpin, let pip resolve."
    )
print("Repo dependencies installed.")


In [ ]:
# torch is only installed as of the previous cell (via the repo's own
# requirements.txt), so device detection lives here rather than in the
# earlier config cell.
import torch
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print("Torch device:", DEVICE)

# Confirmed via dir(data.ps)/dir(data.gf)/dir(data.spot): each module also
# defines a sibling *Weak class for the train_weak split (PlanetScopeStrong
# itself only accepts split in {"train_strong", "test"} — confirmed by its
# own assertion, which is what surfaced this in the first place).
from hub_model import UNetR50                          # noqa: E402
from data.ps import PlanetScopeStrong, PlanetScopeWeak  # noqa: E402
from data.gf import GaofenStrong, GaofenWeak            # noqa: E402
from data.spot import SPOTStrong, SPOTWeak              # noqa: E402


## 3. Fetch a subset of TinyTrees from Hugging Face

TinyTrees is a single 21 GB `tinytrees.zip` — there's no partial-file
download available, so the first `hf_hub_download` call below will pull the
whole archive once (it's cached afterward). What we *do* avoid is unpacking
all 21 GB to disk: we open the zip and selectively extract only the tiles we
sampled, plus each split's `points.gpkg` (small, and needed in full since it
covers every tile in that split).

**Re-running this section is safe and cheap once data is staged.** Before
touching the network or the zip at all, it checks whether every
region/split combination already has tiles on disk under `DATA_DIR` (or, for
`spot/train_weak`, a `.no_imagery` marker recording that there's nothing to
fetch there) — if so, it skips `hf_hub_download` and the extraction loop
entirely. Without this check, re-running would call `random.sample` again
and extract an *additional* random batch of tiles on top of what's already
there, rather than reusing the same ones. Set `FORCE_REFETCH = True` in the
config cell (Section 0's imports/paths cell) if you deliberately want a
fresh random subset.


In [ ]:
from huggingface_hub import hf_hub_download


def region_split_staged(region_key, split):
    """True if this region/split has already been fetched locally — either
    real tiles are present, or a previous run already determined (and
    recorded via the .no_imagery marker) that this split has no bundled
    imagery to fetch (e.g. spot/train_weak).
    """
    d = DATA_DIR / region_key / split
    if not d.exists():
        return False
    if (d / ".no_imagery").exists():
        return True
    return any(d.glob("*.tif"))


needs_fetch = FORCE_REFETCH or not all(
    region_split_staged(region_key, split)
    for region_key in REGIONS
    for split in SPLITS
)

if not needs_fetch:
    print("All region/split combinations already staged locally under "
          f"{DATA_DIR} — skipping Hugging Face download entirely.")
    print("Set FORCE_REFETCH = True in the config cell to force a re-fetch.")
    zip_path = None
else:
    zip_path = hf_hub_download(
        repo_id="dgominski/tinytrees",
        repo_type="dataset",
        filename="tinytrees.zip",
        cache_dir=str(CACHE_DIR),
    )
    print("Zip cached at:", zip_path)


In [ ]:
def sample_and_extract(zf, region_key, split, n_tiles, dest_root):
    """Pick n_tiles .tif entries for region/split and extract them + the
    split's points.gpkg into dest_root, preserving the tinytrees/<region>/<split>/
    layout the loader classes expect.
    """
    prefix = f"tinytrees/{region_key}/{split}/"
    names = zf.namelist()

    tif_names = sorted(n for n in names if n.startswith(prefix) and n.endswith(".tif"))
    gpkg_names = [n for n in names if n.startswith(prefix) and n.endswith(".gpkg")]

    dest_split_dir = dest_root / region_key / split
    dest_split_dir.mkdir(parents=True, exist_ok=True)

    if not tif_names:
        print(f"  [skip] no imagery found for {region_key}/{split} "
              f"(expected for spot/train_weak — imagery lives in Open-Canopy, not TinyTrees)")
        (dest_split_dir / ".no_imagery").touch()  # so future runs don't re-check the zip for this
        return []

    chosen = random.sample(tif_names, min(n_tiles, len(tif_names)))

    for member in chosen + gpkg_names:
        target = dest_root / member[len("tinytrees/"):]
        if target.exists():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(member) as src, open(target, "wb") as out:
            out.write(src.read())

    print(f"  [ok] {region_key}/{split}: extracted {len(chosen)} tiles "
          f"(of {len(tif_names)} available) + {len(gpkg_names)} gpkg file(s)")
    return chosen


if zip_path is None:
    print("Nothing to extract — using what's already staged at", DATA_DIR)
else:
    with zipfile.ZipFile(zip_path) as zf:
        for region_key in REGIONS:
            for split in SPLITS:
                if not FORCE_REFETCH and region_split_staged(region_key, split):
                    print(f"  [already staged] {region_key}/{split} — skipping")
                    continue
                sample_and_extract(zf, region_key, split, N_TILES_PER_SPLIT, DATA_DIR)

print("\nStaged subset at:", DATA_DIR)


## 4. Helper functions

- `tensor_to_rgb_thumbnail`: builds the FiftyOne-visible PNG **from the exact
  tensor the loader returns for that sample** — not from a fresh read of the
  raw `.tif`. This matters: the loader classes crop every tile to a fixed
  `IMSIZE`×`IMSIZE` window (random crop for train splits, center crop for
  test), so if we generated the display thumbnail from the full uncropped
  tile instead, the ground-truth points would be computed on one crop while
  the image shown is a different region entirely — points floating in the
  wrong place, silently. Deriving the thumbnail from the same `image` tensor
  used for `count_map` guarantees they always refer to the same pixels. The
  tensor is already channel-normalized (mean/std) by the loader, but a
  per-channel percentile stretch is scale-invariant, so this still produces a
  reasonable-looking RGB approximation without needing to know the loader's
  internal normalization constants.
- `tile_lonlat_centroid`: reprojects the *whole tile's* centroid to EPSG:4326
  for the Map panel, using `rasterio` — this is a tile-level approximation
  (not the centroid of the specific crop window used for that sample), which
  is fine for "which region is this near" but not for sub-tile precision.
- `points_from_count_map`: given the loader's `count_map` for a tile (a
  sparse 0/1 map of tree locations, on the *same cropped pixel grid* as
  `image`), returns normalized `[x, y]` points for `fo.Keypoint`.
- `get_tile_path`: lookup of the source `.tif` path for sample `i`, used
  only for naming and the tile-centroid geolocation above — never for pixel
  data. Confirmed against a live `PlanetScopeStrong` instance: it exposes a
  `.tifs` attribute, a flat list of absolute path strings index-aligned with
  `ds[i]`. Assumed (not yet separately confirmed) that `GaofenStrong` and
  `SPOTStrong` follow the same pattern — if either raises here, the
  `AttributeError` message prints that class's actual public attributes so
  the fix is a one-line change.
- `get_encoder_features`: pools the model's deepest encoder feature map into
  a single embedding vector per tile, for the cross-sensor domain-shift view
  in Section 5.


In [ ]:
import rasterio
from rasterio.warp import transform_bounds

# ASSUMPTION: channels 0,1,2 of the loader's returned tensor approximate an
# RGB-ish composite. This indexes into the tensor's channel dim (0-indexed),
# which should follow the same band order as the source GeoTIFF. Verify per
# sensor and adjust if the thumbnails look wrong — this only affects display.
CHANNEL_RGB_INDEX = {"ps": (0, 1, 2), "gf": (0, 1, 2), "spot": (0, 1, 2)}


def tensor_to_rgb_thumbnail(image_tensor, region_key, out_path, size=512):
    arr = image_tensor
    if hasattr(arr, "numpy"):
        arr = arr.detach().cpu().numpy()
    idx = CHANNEL_RGB_INDEX[region_key]
    rgb = np.stack([arr[c] for c in idx], axis=-1).astype(np.float32)

    lo, hi = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)
    rgb = (rgb * 255).astype(np.uint8)

    from PIL import Image
    img = Image.fromarray(rgb).resize((size, size), Image.NEAREST)
    img.save(out_path)
    return out_path


def tile_lonlat_centroid(tif_path):
    with rasterio.open(tif_path) as src:
        bounds = src.bounds
        lon_min, lat_min, lon_max, lat_max = transform_bounds(src.crs, "EPSG:4326", *bounds)
    return [(lon_min + lon_max) / 2, (lat_min + lat_max) / 2]


def points_from_count_map(count_map):
    """count_map: (1, H, W) tensor/array of 0/1 tree locations -> normalized [x, y] points."""
    arr = count_map.squeeze()
    if hasattr(arr, "numpy"):
        arr = arr.numpy()
    h, w = arr.shape
    rows, cols = np.nonzero(arr)
    xs = (cols + 0.5) / w
    ys = (rows + 0.5) / h
    return list(zip(xs.tolist(), ys.tolist()))


def get_tile_path(ds, i, region_key, split):
    """Source .tif path for sample i — used only for naming/geolocation,
    never for pixels. Confirmed against a live PlanetScopeStrong instance:
    `ds.tifs` is a flat list of absolute path strings, index-aligned with
    `ds[i]`. Same attribute name is assumed for GaofenStrong/SPOTStrong
    (same base class pattern) — if either of those raises here, that
    assumption was wrong for that class specifically; the AttributeError
    below will still tell you what's actually available on it.
    """
    if hasattr(ds, "tifs"):
        return Path(ds.tifs[i])
    public_attrs = [a for a in dir(ds) if not a.startswith("_")]
    raise AttributeError(
        f"{type(ds).__name__} has no 'tifs' attribute. "
        f"Its public attributes are: {public_attrs}. Update get_tile_path() "
        "to use whichever of these actually holds tile filenames/paths."
    )


_encoder_warned = set()  # so the fallback warning below prints once per model, not once per tile


def get_encoder_features(model, image_tensor):
    """Returns a pooled embedding vector, or None if the encoder submodule
    isn't where we expect. None is filtered out by the caller — a failure
    here should never block dataset/heatmap/count building, since the
    embeddings view is a bonus, not the core demo.

    Confirmed via `list(model.named_children())` on a loaded `dgominski/
    TinyTrees` checkpoint: `UNetR50` wraps a segmentation_models_pytorch
    `Unet` at `.unet`, whose `.encoder` is a `ResNetEncoder` returning a list
    of per-stage feature maps when called directly — `[-1]` is the deepest
    stage (2048-channel, from ResNet-50's layer4). The other paths are kept
    as a defensive fallback only (e.g. in case a differently-configured
    checkpoint, like a swint/vit backbone variant, wraps things differently).
    """
    x = image_tensor.unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        for path in ("unet.encoder", "encoder", "model.encoder", "backbone"):
            obj = model
            try:
                for attr in path.split("."):
                    obj = getattr(obj, attr)
                feats = obj(x)[-1]
                return feats.mean(dim=(2, 3)).squeeze(0).cpu().numpy()
            except AttributeError:
                continue
    if id(model) not in _encoder_warned:
        _encoder_warned.add(id(model))
        print(f"  [warn] couldn't find an encoder submodule on {type(model).__name__} "
              f"via any of the guessed paths — run list(model.named_children()) to find "
              "the real one. Skipping embeddings for this model; heatmaps/counts unaffected.")
    return None


## 5. Build the dataset, run TreeMatch, and collect embeddings — in one pass

Everything downstream of a single tile — the display thumbnail, the
ground-truth points, the predicted density heatmap, the count-error fields,
and the embedding used for the domain-shift view — is now built from **one
`ds[i]` call**, not three separate loops that each re-instantiate the loader
and re-index into it. That matters here specifically because `train_strong`/
`train_weak` splits use a *random* crop per tile: calling `ds[i]` a second
time from a freshly-constructed loader instance would only line up with the
first call if albumentations' per-`Compose` seeding reproduces the exact same
crop sequence across separate instantiations — a real but fragile assumption
not worth carrying into a demo when it's just as easy to not need it at all.

> Dependency-order note: computing the embeddings visualization re-upgrades
> `scikit-learn` past the repo's `requirements.txt` pin (1.3.2), because
> that pin is too old for the `umap-learn` version that came bundled with
> `fiftyone.brain` back in Section 0. This notebook doesn't call sklearn
> directly anywhere, so the upgrade is scoped to fixing that one
> incompatibility, not undoing Section 2's install generally.


In [ ]:
models = {}
for region_key, region in REGIONS.items():
    try:
        model = UNetR50.from_pretrained("dgominski/TinyTrees", subfolder=region["hub_subfolder"])
        model.eval().to(DEVICE)
        models[region_key] = model
        print(f"Loaded TreeMatch checkpoint for {region_key}")
    except Exception as e:
        print(f"  [warn] couldn't load checkpoint for {region_key}: {e}")


In [ ]:
@torch.no_grad()
def run_inference(model, image_tensor):
    """image_tensor: (C+1, H, W) as returned by the loader (spectral bands +
    validity channel already concatenated, per the loader's documented
    return signature). Returns a (H, W) numpy density map.
    """
    x = image_tensor.unsqueeze(0).float().to(DEVICE)
    density = model(x)
    return density.squeeze().cpu().numpy()


In [ ]:
# train_strong/test use the *Strong loader; train_weak uses the sibling
# *Weak loader (see Section 2 import comment). CONFIRMED via PlanetScopeWeak's
# traceback: *Weak has a DIFFERENT constructor shape than *Strong —
#   *Strong(imsize, split, root)   where root = region dir, split appended internally
#   *Weak(imsize, root)            where root = split dir directly, no split arg,
#                                   and it reads f"{root}/points.gpkg" (no split subfolder)
# The instantiation call below branches on this.
LOADER_CLASSES = {
    "ps":   {"train_strong": PlanetScopeStrong, "test": PlanetScopeStrong, "train_weak": PlanetScopeWeak},
    "gf":   {"train_strong": GaofenStrong,       "test": GaofenStrong,       "train_weak": GaofenWeak},
    "spot": {"train_strong": SPOTStrong,         "test": SPOTStrong,         "train_weak": SPOTWeak},
}

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME, persistent=True)

embeddings = []
sample_ids = []

for region_key, region in REGIONS.items():
    model = models.get(region_key)

    for split in SPLITS:
        loader_cls = LOADER_CLASSES[region_key][split]
        split_dir = DATA_DIR / region_key / split
        if not split_dir.exists() or not any(split_dir.glob("*.tif")):
            print(f"[skip] {region_key}/{split}: no imagery on disk "
                  f"(expected for spot/train_weak — see Section 9 notes)")
            continue

        try:
            if split == "train_weak":
                # *Weak reads f"{root}/points.gpkg" directly — root must be the
                # split dir itself, not the region dir.
                ds = loader_cls(imsize=IMSIZE, root=str(split_dir))
            else:
                ds = loader_cls(imsize=IMSIZE, split=split, root=str(DATA_DIR / region_key))
        except (AssertionError, TypeError) as e:
            print(f"[skip] {region_key}/{split}: {loader_cls.__name__} rejected "
                  f"this call ({e!r}) — constructor shape differs from what's assumed here.")
            continue
        n = len(ds)
        print(f"{region_key}/{split}: {n} tiles")

        for i in range(n):
            image, valid_mask, count_map = ds[i]
            tif_path = get_tile_path(ds, i, region_key, split)  # raises loudly if the guess is wrong

            thumb_path = MEDIA_DIR / f"{region_key}_{split}_{tif_path.stem}_{i}.png"
            tensor_to_rgb_thumbnail(image, region_key, thumb_path)

            points = points_from_count_map(count_map)
            lonlat = tile_lonlat_centroid(tif_path)  # tile centroid — see Section 4 caveat

            sample = fo.Sample(filepath=str(thumb_path))
            sample["ground_truth_points"] = fo.Keypoints(
                keypoints=[fo.Keypoint(label="tree", points=points)]
            )
            sample["region"] = region["name"]
            sample["sensor"] = region["sensor"]
            sample["split"] = split
            sample["count_gt"] = len(points)
            sample["tile_name"] = tif_path.name
            sample["location"] = fo.GeoLocation(point=lonlat)
            sample.tags = [region["name"], split]

            if model is not None:
                density = run_inference(model, image)
                count_pred = float(density.sum())

                density_png = MEDIA_DIR / f"{region_key}_{split}_{tif_path.stem}_{i}_density.png"
                norm = (density - density.min()) / (density.max() - density.min() + 1e-8)
                from PIL import Image
                Image.fromarray((norm * 255).astype(np.uint8)).save(density_png)

                sample["density_pred"] = fo.Heatmap(map_path=str(density_png))
                sample["count_pred"] = count_pred
                sample["count_error"] = count_pred - sample["count_gt"]
                sample["count_abs_error"] = abs(count_pred - sample["count_gt"])

                emb = get_encoder_features(model, image)
                if emb is not None:
                    embeddings.append(emb)
                    sample_ids.append(None)  # placeholder, filled in after add_sample below

            dataset.add_sample(sample)
            if model is not None and sample_ids and sample_ids[-1] is None:
                sample_ids[-1] = sample.id

embeddings = np.stack(embeddings) if embeddings else np.empty((0, 0))
print()
print(dataset)
print("Embeddings shape:", embeddings.shape)


In [ ]:
if embeddings.shape[0] > 0:
    # Section 2's `pip install -r requirements.txt` force-downgrades
    # scikit-learn to the repo's exact pin (1.3.2), which breaks the newer
    # umap-learn that came bundled with fiftyone.brain back in Section 0 —
    # newer umap-learn passes check_array(..., ensure_all_finite=...), a
    # kwarg that didn't exist before sklearn 1.6. Nothing in this notebook
    # calls sklearn directly (only umap-learn/fiftyone.brain do, internally),
    # so it's safe to re-upgrade it here without affecting anything else.
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", "scikit-learn>=1.6"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("Failed to upgrade scikit-learn for umap-learn compatibility.")

    emb_view = dataset.select(sample_ids, ordered=True)
    fob.compute_visualization(
        emb_view,
        embeddings=embeddings,
        method="umap",
        brain_key="sensor_domain_viz",
    )
    print("Open the Embeddings panel in the App and pick brain_key='sensor_domain_viz', "
          "then color by 'sensor' or 'region' to see the clustering.")
else:
    print("No embeddings computed — check that model checkpoints loaded above, "
          "and that get_tile_path() didn't raise (see its error message for the fix).")


## 6. Curated views + launch the App

A few views worth having ready for a live demo:

- **Weak vs. strong, same region** — flip between `split == train_weak` and
  `split == train_strong` on the same region to visually show how much
  noisier the pseudolabels are.
- **Worst count error** — sort descending by `count_abs_error` to jump
  straight to the model's failure modes.
- **Per-region / per-sensor slice** — tag-based filtering to isolate one
  sensor at a time.


In [ ]:
weak_vs_strong_rwanda_weak = dataset.match(F("region") == "rwanda").match(F("split") == "train_weak")
weak_vs_strong_rwanda_strong = dataset.match(F("region") == "rwanda").match(F("split") == "train_strong")

worst_errors = dataset.exists("count_abs_error").sort_by("count_abs_error", reverse=True)

print("Weak (Rwanda):", len(weak_vs_strong_rwanda_weak))
print("Strong (Rwanda):", len(weak_vs_strong_rwanda_strong))
print("Worst predictions:", len(worst_errors))


Saving these as named views means they persist on the dataset across
sessions and show up in the App's view sidebar for one-click switching —
useful for a live demo where you don't want to be re-typing `match()` calls
on the spot. `save_view()` is idempotent here (existing views with the same
name get overwritten), so this cell is safe to re-run after adding more
tiles or regions.


In [ ]:
def save_or_update_view(dataset, name, view):
    if dataset.has_saved_view(name):
        dataset.delete_saved_view(name)
    dataset.save_view(name, view)
    print(f"  [saved] {name}  ({len(view)} samples)")


# 1. Weak vs. strong, per region — flip between the two to see how much
#    noisier the pseudolabels are on the same tiles/region.
for region_key, region in REGIONS.items():
    region_name = region["name"]
    weak_view = dataset.match(F("region") == region_name).match(F("split") == "train_weak")
    strong_view = dataset.match(F("region") == region_name).match(F("split") == "train_strong")
    if len(weak_view) > 0:
        save_or_update_view(dataset, f"{region_name}_train_weak", weak_view)
    if len(strong_view) > 0:
        save_or_update_view(dataset, f"{region_name}_train_strong", strong_view)

# 2. Worst count error — the model's failure modes, sorted to the top.
worst_errors = dataset.exists("count_abs_error").sort_by("count_abs_error", reverse=True)
if len(worst_errors) > 0:
    save_or_update_view(dataset, "worst_count_error", worst_errors)

# 3. Per-region / per-sensor slices — isolate one sensor at a time.
for region_key, region in REGIONS.items():
    region_view = dataset.match(F("region") == region["name"])
    if len(region_view) > 0:
        save_or_update_view(dataset, f"region_{region['name']}", region_view)

print()
print("Saved views:", dataset.list_saved_views())


In [ ]:
session = fo.launch_app(dataset)

# Jump straight to the most interesting view for a live demo:
session.view = worst_errors


## 7. Notes, caveats, and next steps

- **License**: TinyTrees imagery is CC BY-NC 4.0 — keep this demo to
  research/education contexts, don't ship it commercially.
- **Band order for thumbnails**: `CHANNEL_RGB_INDEX` in Section 4 is a
  guess at which 3 channels of the loader's tensor look RGB-ish. If the
  images look wrong (inverted colors, too dark/bright), adjust it — it only
  affects display, not model correctness (the model consumes the same
  tensor directly, at full channel count).
- **IMSIZE**: the loader classes always crop to a fixed square (`IMSIZE` in
  the config cell, default 256) — there's no "give me the full tile"
  option. Must stay divisible by 32.
- **`get_tile_path` and encoder access**: flagged inline with
  `# TODO: verify against repo source`. Once you have the repo cloned
  locally, a two-minute read of `data/ps.py` and `hub_model.py` will confirm
  or correct these guesses.
- **Scaling up**: bump `N_TILES_PER_SPLIT`, or drop the sampling entirely and
  extract every tile for a full-scale dataset — the pipeline doesn't change,
  just the runtime and disk footprint.
- **SPOT weak split**: pseudolabels are bundled, but the SPOT-6 imagery
  itself isn't (per the repo's README, it comes from the separate
  Open-Canopy dataset). This notebook skips building samples for
  `spot/train_weak` unless you separately download Open-Canopy and point
  `spot_imagery_root` at it, per the repo's own instructions.
- **Persisting**: the dataset was created with `persistent=True`, so it'll
  still be there (`fo.load_dataset("tinytrees-demo")`) the next time you
  start FiftyOne — no need to rebuild it from scratch every session.
